In [55]:
import pandas as pd

Połączenie danych z MovieLens i z TMDB pobramych dla zbioru MovieLens

In [56]:
movie_lens = pd.read_csv('../data/merged/movielens.csv')
movie_lens

,movieId,imdbId,tmdbId,title,year,genres,top_5_tags
0,1,114709,862.0,Toy Story,1995.0,Adventure|Animation|Children|Comedy|Fantasy,"toys, computer animation, pixar animation, ani..."
1,2,113497,8844.0,Jumanji,1995.0,Adventure|Children|Fantasy,"adventure, children, fantasy, kids, childhood"
2,3,113228,15602.0,Grumpier Old Men,1995.0,Comedy|Romance,"good sequel, sequel, sequels, comedy, original"
3,4,114885,31357.0,Waiting to Exhale,1995.0,Comedy|Drama|Romance,"women, chick flick, girlie movie, romantic, di..."
4,5,113041,11862.0,Father of the Bride Part II,1995.0,Comedy,"good sequel, sequel, sequels, midlife crisis, ..."
...,...,...,...,...,...,...,...
86532,288967,14418234,845861.0,State of Siege: Temple Attack,2021.0,Action|Drama,NaN
86533,288971,11162178,878958.0,Ouija Japan,2021.0,Action|Horror,NaN
86534,288975,70199,150392.0,The Men Who Made the Movies: Howard Hawks,1973.0,Documentary,NaN
86535,288977,23050520,1102551.0,Skinford: Death Sentence,2023.0,Crime|Thriller,NaN


In [57]:
tmdb = pd.read_csv('../data/tmdb/tmdb_data_from_movieLens.csv')
tmdb

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,genres,origin_countries,spoken_languages
0,862.0,Toy Story,30000000,401157969,1995-11-22,81,en,21.5820,7.971,19651,"Family, Comedy, Animation, Adventure",United States of America,English
1,8844.0,Jumanji,65000000,262821940,1995-12-15,104,en,3.1836,7.245,11172,"Adventure, Fantasy, Family",United States of America,"English, Français"
2,15602.0,Grumpier Old Men,25000000,71518503,1995-12-22,101,en,2.1246,6.500,420,"Romance, Comedy",United States of America,English
3,31357.0,Waiting to Exhale,16000000,81452156,1995-12-22,127,en,2.0018,6.253,194,"Comedy, Drama, Romance",United States of America,English
4,11862.0,Father of the Bride Part II,0,76594107,1995-12-08,106,en,2.5346,6.275,795,"Comedy, Family",United States of America,English
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57703,385377.0,Ballet of Blood,0,0,2016-03-01,98,en,0.3408,1.800,10,Horror,United States of America,English
57704,644612.0,Letters Of Happiness,0,0,2019-12-03,103,ru,0.7063,6.300,9,"Drama, Family",Russia,Pусский
57705,845861.0,Akshardham: Operation Vajra Shakti,0,0,2025-07-04,106,hi,1.2825,10.000,2,"Action, Drama",India,हिन्दी
57706,878958.0,Ouija Japan,0,0,2021-10-19,78,en,0.6105,2.000,1,"Action, Horror",NaN,"日本語, English"


In [58]:
df = pd.merge(
    tmdb,
    movie_lens.drop(columns=['movieId', 'title']),
    how='left',
    on='tmdbId').rename(columns={'top_5_tags': 'keywords'})
df['release_date'] = pd.to_datetime(df['release_date'])
df

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,genres_x,origin_countries,spoken_languages,imdbId,year,genres_y,keywords
0,862.0,Toy Story,30000000,401157969,1995-11-22,81,en,21.5820,7.971,19651,"Family, Comedy, Animation, Adventure",United States of America,English,114709,1995.0,Adventure|Animation|Children|Comedy|Fantasy,"toys, computer animation, pixar animation, ani..."
1,8844.0,Jumanji,65000000,262821940,1995-12-15,104,en,3.1836,7.245,11172,"Adventure, Fantasy, Family",United States of America,"English, Français",113497,1995.0,Adventure|Children|Fantasy,"adventure, children, fantasy, kids, childhood"
2,15602.0,Grumpier Old Men,25000000,71518503,1995-12-22,101,en,2.1246,6.500,420,"Romance, Comedy",United States of America,English,113228,1995.0,Comedy|Romance,"good sequel, sequel, sequels, comedy, original"
3,31357.0,Waiting to Exhale,16000000,81452156,1995-12-22,127,en,2.0018,6.253,194,"Comedy, Drama, Romance",United States of America,English,114885,1995.0,Comedy|Drama|Romance,"women, chick flick, girlie movie, romantic, di..."
4,11862.0,Father of the Bride Part II,0,76594107,1995-12-08,106,en,2.5346,6.275,795,"Comedy, Family",United States of America,English,113041,1995.0,Comedy,"good sequel, sequel, sequels, midlife crisis, ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57733,385377.0,Ballet of Blood,0,0,2016-03-01,98,en,0.3408,1.800,10,Horror,United States of America,English,3061100,2016.0,Horror,NaN
57734,644612.0,Letters Of Happiness,0,0,2019-12-03,103,ru,0.7063,6.300,9,"Drama, Family",Russia,Pусский,12073988,2019.0,Children|Drama,NaN
57735,845861.0,Akshardham: Operation Vajra Shakti,0,0,2025-07-04,106,hi,1.2825,10.000,2,"Action, Drama",India,हिन्दी,14418234,2021.0,Action|Drama,NaN
57736,878958.0,Ouija Japan,0,0,2021-10-19,78,en,0.6105,2.000,1,"Action, Horror",NaN,"日本語, English",11162178,2021.0,Action|Horror,NaN


In [59]:
genres1 = df['genres_x'].str.lower().str.replace(' ', '')
genres2 = df['genres_y'].str.lower().str.replace('|', ',')
df['genres'] = (
    (genres1 + ',' + genres2)
    .str.split(',')
    .explode()
    .dropna()
    .str.strip()
    .loc[lambda x: x != '']
    .groupby(level=0)
    .unique()
    .apply(lambda x: ','.join(x))
)
df.drop(['genres_x', 'genres_y', 'imdbId'], axis=1, inplace=True)
df['genres']

0        family,comedy,animation,adventure,children,fan...
1                        adventure,fantasy,family,children
2                                           romance,comedy
3                                     comedy,drama,romance
4                                            comedy,family
                               ...                        
57733                                               horror
57734                                drama,family,children
57735                                         action,drama
57736                                        action,horror
57737                                          documentary
Name: genres, Length: 57738, dtype: object

Połączenie z dodatkowo pobranymi filmami z bazy TMDB, których nie było w MovieLens

In [60]:
tmdb1 = pd.read_csv('../data/tmdb/tmdb_data.csv')
tmdb1['keywords'] = tmdb1['keywords'].str.replace(', ', ',')
tmdb1['genres'] = tmdb1['genres'].str.replace(', ', ',').str.lower()
tmdb1['release_date'] = pd.to_datetime(tmdb1['release_date'])
tmdb1['year'] = tmdb1['release_date'].dt.year
tmdb1

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,origin_countries,spoken_languages,keywords,genres,year
0,581,Dances with Wolves,22000000,424208848,1990-03-30,181,en,9.5110,7.834,4613,"United Kingdom, United States of America",English,"friendship,mutiny,countryside,based on novel o...","adventure,drama,western",1990
1,60898,Erotic Ghost Story,0,0,1990-05-19,88,cn,8.2792,6.200,93,Hong Kong,广州话 / 廣州話,NaN,"fantasy,drama,horror",1990
2,431418,Carnaval,0,0,1990-02-07,26,pt,18.2781,0.000,0,"United Kingdom, Brazil, Portugal, France, Ital...",Português,"rio de janeiro,carnival",drama,1990
3,196,Back to the Future Part III,40000000,244527583,1990-05-25,119,en,7.5005,7.490,11365,United States of America,English,"california,saloon,sports car,inventor,horsebac...","adventure,comedy,science fiction",1990
4,114,Pretty Woman,14000000,463406268,1990-03-23,120,en,8.3400,7.451,8874,United States of America,English,"prostitute,sports car,capitalism,expensive res...","romance,comedy",1990
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35995,1591516,Evil Influencer: The Jodi Hildebrandt Story,0,0,2025-12-30,102,en,2.5060,6.250,44,United States of America,English,"true crime,crime documentary,disgusted",documentary,2025
35996,1288599,Una famiglia sottosopra,0,0,2025-11-06,100,it,2.7632,6.417,6,Italy,Italiano,NaN,"comedy,fantasy",2025
35997,1238188,The Captive,10720500,0,2025-09-12,134,es,2.2714,6.533,61,"Italy, Spain","العربية, Español","lgbt,gay theme","drama,history",2025
35998,1031084,The Currents,0,0,2025-11-13,104,es,1.8391,5.750,2,"Argentina, Switzerland",Español,NaN,drama,2025


In [61]:
df = pd.concat([df, tmdb1], ignore_index=True)
print(len(df))
df.drop_duplicates(inplace=True)
df

93738


,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,origin_countries,spoken_languages,year,keywords,genres
0,862.0,Toy Story,30000000,401157969,1995-11-22,81,en,21.5820,7.971,19651,United States of America,English,1995.0,"toys, computer animation, pixar animation, ani...","family,comedy,animation,adventure,children,fan..."
1,8844.0,Jumanji,65000000,262821940,1995-12-15,104,en,3.1836,7.245,11172,United States of America,"English, Français",1995.0,"adventure, children, fantasy, kids, childhood","adventure,fantasy,family,children"
2,15602.0,Grumpier Old Men,25000000,71518503,1995-12-22,101,en,2.1246,6.500,420,United States of America,English,1995.0,"good sequel, sequel, sequels, comedy, original","romance,comedy"
3,31357.0,Waiting to Exhale,16000000,81452156,1995-12-22,127,en,2.0018,6.253,194,United States of America,English,1995.0,"women, chick flick, girlie movie, romantic, di...","comedy,drama,romance"
4,11862.0,Father of the Bride Part II,0,76594107,1995-12-08,106,en,2.5346,6.275,795,United States of America,English,1995.0,"good sequel, sequel, sequels, midlife crisis, ...","comedy,family"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93733,1591516.0,Evil Influencer: The Jodi Hildebrandt Story,0,0,2025-12-30,102,en,2.5060,6.250,44,United States of America,English,2025.0,"true crime,crime documentary,disgusted",documentary
93734,1288599.0,Una famiglia sottosopra,0,0,2025-11-06,100,it,2.7632,6.417,6,Italy,Italiano,2025.0,NaN,"comedy,fantasy"
93735,1238188.0,The Captive,10720500,0,2025-09-12,134,es,2.2714,6.533,61,"Italy, Spain","العربية, Español",2025.0,"lgbt,gay theme","drama,history"
93736,1031084.0,The Currents,0,0,2025-11-13,104,es,1.8391,5.750,2,"Argentina, Switzerland",Español,2025.0,NaN,drama


Połączenie danych o ludziach

In [62]:
people = pd.read_csv('../data/tmdb/tmdb_people_movieLens.csv')
print(len(people))
people1 = pd.read_csv('../data/tmdb/tmdb_people.csv')
print(len(people1))
people = pd.concat([people, people1], ignore_index=True)
people

408979
266720


,movie_id,person_id,name,gender,known_for_department,popularity,job
0,862.0,31,Tom Hanks,2,Acting,20.1613,actor
1,862.0,12898,Tim Allen,2,Acting,2.9738,actor
2,862.0,7167,Don Rickles,2,Acting,2.1703,actor
3,862.0,12899,Jim Varney,2,Acting,0.9327,actor
4,862.0,12900,Wallace Shawn,2,Acting,3.2563,actor
...,...,...,...,...,...,...,...
675694,1446765.0,2048825,Volodymyr Rashchuk,2,Acting,0.6362,actor
675695,1446765.0,3190635,Oleksandr Yatsentiuk,2,Acting,0.3420,actor
675696,1446765.0,3108544,Mykhailo Dziuba,2,Acting,0.3992,actor
675697,1446765.0,2703511,Oleksii Taranenko,2,Directing,0.3470,Director


Do każdego filmu po jednym najpopularniejszym reżyserze, scenarzyście i producencie

In [63]:
single_choice_departments = ['Production', 'Writing', 'Directing']
actors = people[people['known_for_department'] == 'Acting']
others = people[people['known_for_department'].isin(single_choice_departments)]
others_top = others.loc[others.groupby(['movie_id', 'known_for_department'])['popularity'].idxmax()]
people = pd.concat([actors, others_top], ignore_index=True)
people['known_for_department'] = people['known_for_department'].replace({
    'Acting': 'actor',
    'Writing': 'writer',
    'Directing': 'director',
    'Production': 'producer',
})
people.drop(columns=['job'], inplace=True)
people.rename(columns={'known_for_department': 'job', 'person_id': 'id'}, inplace=True)
people

,movie_id,id,name,gender,job,popularity
0,862.0,31,Tom Hanks,2,actor,20.1613
1,862.0,12898,Tim Allen,2,actor,2.9738
2,862.0,7167,Don Rickles,2,actor,2.1703
3,862.0,12899,Jim Varney,2,actor,0.9327
4,862.0,12900,Wallace Shawn,2,actor,3.2563
...,...,...,...,...,...,...
553837,1618818.0,1057967,Tadafumi Tomioka,2,director,0.8424
553838,1618818.0,126783,Masato Katō,2,writer,1.2668
553839,1640154.0,6029113,紫堂安二郎,0,director,0.0338
553840,1653378.0,1270590,Akira Fukamachi,2,director,0.8756


Do każdego filmu 5 najpopularniejszych aktorów - piwot danych (1 to najbardziej popularny)

In [64]:
actors = people[people['job'] == 'actor'].copy()
actors['rank'] = actors.groupby('movie_id')['popularity'].rank(method='first', ascending=False)
actors = actors[actors['rank'] <= 5]
actors['rank'] = actors['rank'].astype(int)

rows = []
for movie_id, group in actors.groupby("movie_id"):
    row = {"movie_id": movie_id}
    for i, person in group.iterrows():
        row[f"actor{person['rank']}_id"] = person['id']
        row[f"actor{person['rank']}_name"] = person['name']
        row[f"actor{person['rank']}_popularity"] = person['popularity']
        row[f"actor{person['rank']}_gender"] = person['gender']
    rows.append(row)
actors = pd.DataFrame(rows)
actors

,movie_id,actor2_id,actor2_name,actor2_popularity,actor2_gender,actor3_id,actor3_name,actor3_popularity,actor3_gender,actor5_id,...,actor5_popularity,actor5_gender,actor1_id,actor1_name,actor1_popularity,actor1_gender,actor4_id,actor4_name,actor4_popularity,actor4_gender
0,5.0,3129.0,Tim Roth,4.9629,2.0,3130.0,Jennifer Beals,3.9675,1.0,3125.0,...,2.6797,1.0,3129,Tim Roth,5.6589,2,3130.0,Jennifer Beals,3.7741,1.0
1,6.0,5724.0,Denis Leary,2.9368,2.0,12799.0,Jeremy Piven,2.7546,2.0,9777.0,...,2.5604,2.0,5724,Denis Leary,3.2793,2,9777.0,Cuba Gooding Jr.,2.6821,2.0
2,12.0,5293.0,Willem Dafoe,6.5897,2.0,118.0,Geoffrey Rush,2.4578,2.0,12.0,...,2.0017,2.0,5293,Willem Dafoe,7.2926,2,118.0,Geoffrey Rush,2.2366,2.0
3,13.0,31.0,Tom Hanks,17.5197,2.0,35.0,Sally Field,6.1880,1.0,32.0,...,3.9461,1.0,31,Tom Hanks,20.1613,2,35.0,Sally Field,5.3287,1.0
4,14.0,1979.0,Kevin Spacey,4.5494,2.0,516.0,Annette Bening,3.7232,1.0,8211.0,...,3.0512,1.0,1979,Kevin Spacey,4.7780,2,516.0,Annette Bening,3.4638,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71972,1614973.0,3164807.0,Yoon Yool,8.5631,1.0,2710789.0,Seung Ha,1.9809,1.0,NaN,...,NaN,NaN,1907997,Min Do-yoon,9.2665,2,NaN,NaN,NaN,NaN
71973,1618818.0,553623.0,Misa Aika,1.1727,1.0,96559.0,Kazuya Takahashi,1.1306,2.0,2213136.0,...,0.5715,1.0,128480,Yoichi Nukumizu,1.3237,2,133656.0,Yukijiro Hotaru,1.1037,2.0
71974,1628733.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3164807,Yoon Yool,8.5631,1,NaN,NaN,NaN,NaN
71975,1640154.0,3610063.0,Horiuchi Hajime,0.0000,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2390106,Arina Arata,0.0000,1,NaN,NaN,NaN,NaN


Piwot pozostałych osób

In [65]:
others = people[people['job'] != 'actor'].copy()
rows = []
for movie_id, group in others.groupby("movie_id"):
    row = {"movie_id": movie_id}
    for i, person in group.iterrows():
        row[f"{person['job']}_id"] = person['id']
        row[f"{person['job']}_name"] = person['name']
        row[f"{person['job']}_popularity"] = person['popularity']
        row[f"{person['job']}_gender"] = person['gender']
    rows.append(row)
others = pd.DataFrame(rows)
others

,movie_id,director_id,director_name,director_popularity,director_gender,writer_id,writer_name,writer_popularity,writer_gender,producer_id,producer_name,producer_popularity,producer_gender
0,5.0,138.0,Quentin Tarantino,7.3808,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,6.0,2042.0,Stephen Hopkins,1.5159,2.0,52035.0,Lewis Colick,2.3831,2.0,NaN,NaN,NaN,NaN
2,12.0,13.0,Albert Brooks,2.1655,2.0,7.0,Andrew Stanton,2.1229,2.0,NaN,NaN,NaN,NaN
3,13.0,24.0,Robert Zemeckis,3.0808,2.0,27.0,Eric Roth,2.1400,2.0,NaN,NaN,NaN,NaN
4,14.0,39.0,Sam Mendes,1.9894,2.0,2152.0,Alan Ball,1.3321,2.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
70530,1609471.0,NaN,NaN,NaN,NaN,3209002.0,Ha Seung-hee,0.1172,0.0,NaN,NaN,NaN,NaN
70531,1614826.0,1761154.0,Bruce Hepton,0.4141,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
70532,1618818.0,1057967.0,Tadafumi Tomioka,0.8424,2.0,126783.0,Masato Katō,1.2668,2.0,NaN,NaN,NaN,NaN
70533,1640154.0,6029113.0,紫堂安二郎,0.0338,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Połączenie danych o ludziach i filmach

In [66]:
df = pd.merge(
    left=df,
    right=others,
    how='left',
    left_on='tmdbId',
    right_on='movie_id',
)
df = pd.merge(
    left=df,
    right=actors,
    how='left',
    left_on='tmdbId',
    right_on='movie_id',
)
df = df.drop(columns=['movie_id_x', 'movie_id_y'])
df.rename(columns={'top_5_tags': 'keywords'}, inplace=True)
df['keywords'] = df['keywords'].str.replace(', ', ',')
df

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,...,actor5_popularity,actor5_gender,actor1_id,actor1_name,actor1_popularity,actor1_gender,actor4_id,actor4_name,actor4_popularity,actor4_gender
0,862.0,Toy Story,30000000,401157969,1995-11-22,81,en,21.5820,7.971,19651,...,2.9738,2.0,31.0,Tom Hanks,20.1613,2.0,12898.0,Tim Allen,3.0445,2.0
1,8844.0,Jumanji,65000000,262821940,1995-12-15,104,en,3.1836,7.245,11172,...,2.6595,1.0,205.0,Kirsten Dunst,7.7741,1.0,2157.0,Robin Williams,4.5865,2.0
2,15602.0,Grumpier Old Men,25000000,71518503,1995-12-22,101,en,2.1246,6.500,420,...,3.0548,1.0,16757.0,Sophia Loren,4.9967,1.0,13567.0,Ann-Margret,3.2713,1.0
3,31357.0,Waiting to Exhale,16000000,81452156,1995-12-22,127,en,2.0018,6.253,194,...,2.5071,1.0,2178.0,Forest Whitaker,6.9280,2.0,9780.0,Angela Bassett,3.8558,1.0
4,11862.0,Father of the Bride Part II,0,76594107,1995-12-08,106,en,2.5346,6.275,795,...,2.5075,2.0,3092.0,Diane Keaton,3.9430,1.0,67773.0,Steve Martin,3.5511,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93710,1591516.0,Evil Influencer: The Jodi Hildebrandt Story,0,0,2025-12-30,102,en,2.5060,6.250,44,...,0.0168,0.0,4925066.0,Ruby Franke,0.1695,1.0,5933553.0,Ethan Prete,0.0461,0.0
93711,1288599.0,Una famiglia sottosopra,0,0,2025-11-06,100,it,2.7632,6.417,6,...,0.1904,0.0,36322.0,Valentina Lodovini,1.9751,1.0,4707103.0,Martina Bernocchi,0.4787,0.0
93712,1238188.0,The Captive,10720500,0,2025-09-12,134,es,2.2714,6.533,61,...,0.7828,2.0,81848.0,Miguel Rellán,1.8415,2.0,43321.0,Fernando Tejero,1.2750,2.0
93713,1031084.0,The Currents,0,0,2025-11-13,104,es,1.8391,5.750,2,...,0.0000,0.0,1086358.0,Esteban Bigliardi,0.7186,2.0,2355348.0,Isabel Aimé González Sola,0.3136,0.0


Uwzględnienie inflacji - dane są w dolarach amerykańskich, a więc uwzględniamy inflację tej waluty. Jako rok bazowy przyjmujemy 2025 rok i pod ten rok dostosowujemy wszystkie wartości.

In [67]:
cpi = pd.read_excel('../data/cpi.xlsx', skiprows=11)
cpi = cpi[['Year', 'Annual']].set_index('Year').to_dict()['Annual']
cpi

C:\Users\Gabi\PycharmProjects\Data_Exploration\venv\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


{1990: 130.7,
 1991: 136.2,
 1992: 140.3,
 1993: 144.5,
 1994: 148.2,
 1995: 152.4,
 1996: 156.9,
 1997: 160.5,
 1998: 163.0,
 1999: 166.6,
 2000: 172.2,
 2001: 177.1,
 2002: 179.9,
 2003: 184.0,
 2004: 188.9,
 2005: 195.3,
 2006: 201.6,
 2007: 207.342,
 2008: 215.303,
 2009: 214.537,
 2010: 218.056,
 2011: 224.939,
 2012: 229.594,
 2013: 232.957,
 2014: 236.736,
 2015: 237.017,
 2016: 240.007,
 2017: 245.12,
 2018: 251.107,
 2019: 255.657,
 2020: 258.811,
 2021: 270.97,
 2022: 292.655,
 2023: 304.702,
 2024: 313.689,
 2025: 321.943,
 2026: nan}

In [85]:
base_year = 2025
data = df.copy()
data['budget_adjusted'] = round(df['budget'] * (cpi[base_year] / data['year'].map(cpi)), 2)
data['revenue_adjusted'] = round(df['revenue'] * (cpi[base_year] / data['year'].map(cpi)), 2)
data

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,...,actor1_id,actor1_name,actor1_popularity,actor1_gender,actor4_id,actor4_name,actor4_popularity,actor4_gender,budget_adjusted,revenue_adjusted
0,862.0,Toy Story,30000000,401157969,1995-11-22,81,en,21.5820,7.971,19651,...,31.0,Tom Hanks,20.1613,2.0,12898.0,Tim Allen,3.0445,2.0,6.337461e+07,8.474409e+08
1,8844.0,Jumanji,65000000,262821940,1995-12-15,104,en,3.1836,7.245,11172,...,205.0,Kirsten Dunst,7.7741,1.0,2157.0,Robin Williams,4.5865,2.0,1.373116e+08,5.552079e+08
2,15602.0,Grumpier Old Men,25000000,71518503,1995-12-22,101,en,2.1246,6.500,420,...,16757.0,Sophia Loren,4.9967,1.0,13567.0,Ann-Margret,3.2713,1.0,5.281217e+07,1.510819e+08
3,31357.0,Waiting to Exhale,16000000,81452156,1995-12-22,127,en,2.0018,6.253,194,...,2178.0,Forest Whitaker,6.9280,2.0,9780.0,Angela Bassett,3.8558,1.0,3.379979e+07,1.720666e+08
4,11862.0,Father of the Bride Part II,0,76594107,1995-12-08,106,en,2.5346,6.275,795,...,3092.0,Diane Keaton,3.9430,1.0,67773.0,Steve Martin,3.5511,2.0,0.000000e+00,1.618040e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93710,1591516.0,Evil Influencer: The Jodi Hildebrandt Story,0,0,2025-12-30,102,en,2.5060,6.250,44,...,4925066.0,Ruby Franke,0.1695,1.0,5933553.0,Ethan Prete,0.0461,0.0,0.000000e+00,0.000000e+00
93711,1288599.0,Una famiglia sottosopra,0,0,2025-11-06,100,it,2.7632,6.417,6,...,36322.0,Valentina Lodovini,1.9751,1.0,4707103.0,Martina Bernocchi,0.4787,0.0,0.000000e+00,0.000000e+00
93712,1238188.0,The Captive,10720500,0,2025-09-12,134,es,2.2714,6.533,61,...,81848.0,Miguel Rellán,1.8415,2.0,43321.0,Fernando Tejero,1.2750,2.0,1.072050e+07,0.000000e+00
93713,1031084.0,The Currents,0,0,2025-11-13,104,es,1.8391,5.750,2,...,1086358.0,Esteban Bigliardi,0.7186,2.0,2355348.0,Isabel Aimé González Sola,0.3136,0.0,0.000000e+00,0.000000e+00


Usuwanie duplikatów tych samych tytułów, tmdbId i roku produkcji

In [86]:
data['keywords_count'] = data['keywords'].fillna('').str.count(',')
data = data.sort_values(by='keywords_count', ascending=True)
data = data.drop(columns=['keywords_count'])

data = data.drop_duplicates(subset=['tmdbId', 'title', 'release_date'], keep='first')
data

,tmdbId,title,budget,revenue,release_date,runtime,original_language,popularity,vote_average,vote_count,...,actor1_id,actor1_name,actor1_popularity,actor1_gender,actor4_id,actor4_name,actor4_popularity,actor4_gender,budget_adjusted,revenue_adjusted
60585,462294.0,Second to None,0,0,1992-10-08,91,cn,0.7994,5.000,1,...,227778.0,Carol Cheng,2.1777,1.0,96890.0,Bonnie Fu Yuk-Jing,1.4402,1.0,0.00,0.000000e+00
60587,427991.0,Teamster Boss: The Jackie Presser Story,0,0,1992-09-12,90,en,0.9740,5.250,5,...,6197.0,Brian Dennehy,2.6311,2.0,3265.0,Eli Wallach,2.2118,2.0,0.00,0.000000e+00
60588,581509.0,Bitter Harvest,0,0,1992-07-22,75,en,1.3639,0.000,0,...,75604.0,Yul Vazquez,2.3795,2.0,21708.0,Tomas Milian,0.9317,2.0,0.00,0.000000e+00
60589,94361.0,Shooting Elizabeth,0,0,1992-11-18,96,en,1.0029,5.300,11,...,4785.0,Jeff Goldblum,4.5287,2.0,24496.0,Simón Andreu,0.6623,2.0,0.00,0.000000e+00
60590,466687.0,Sosuke Loses His Lover,0,0,1992-10-02,106,ja,1.1569,2.000,2,...,36074.0,Kazuko Yoshiyuki,1.7328,1.0,1048658.0,Miwako Fujitani,0.7130,1.0,0.00,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91494,988402.0,Humanist Vampire Seeking Consenting Suicidal P...,3000000,97578,2023-10-13,91,fr,2.3394,7.400,367,...,1661094.0,Noémie O'Farrell,0.9632,1.0,2372688.0,Sara Montpetit,0.4544,1.0,3169749.46,1.030993e+05
93285,1259102.0,Eternity,12000000,32867848,2025-11-26,114,en,17.7926,7.000,540,...,550843.0,Elizabeth Olsen,7.0428,1.0,1180099.0,Da'Vine Joy Randolph,1.6653,1.0,12000000.00,3.286785e+07
71127,79610.0,Behind the Red Door,0,0,2003-01-12,105,en,0.9649,5.300,26,...,2628.0,Kiefer Sutherland,3.8443,2.0,63545.0,Hannah Lochner,1.1395,1.0,0.00,0.000000e+00
80984,104755.0,The Lords of Salem,1500000,1165882,2013-04-18,101,en,1.8678,5.444,915,...,41229.0,Meg Foster,4.8900,1.0,21319.0,Sheri Moon Zombie,0.9774,1.0,2072976.99,1.611231e+06


In [87]:
len(data[(data['revenue'] > 0)])

13990

In [88]:
data.to_csv('../data/merged/merged_data.csv')